# Intrinsic-Dimension Accuracy Stress Test — LHSD-inspired vs Projector-Aware Noise-Scale Selection

**Research question:** With score approximation error, can the projector-aware selector reduce **intrinsic-dimension MAE**, not only tangent-projector error?

**Essential experimental controls**
- The same analytically Gaussian-smoothed score family and **constructed conservative score errors** are used for both methods.
- Both use the same nine Gaussian convolution scales and the **same Hill-filter dimension estimator** (\(c=0.1, p=4\)). Only the scale-selection rule differs.
- Selector parameters are copied *without modification* from the earlier benchmark; no test-label tuning. True sphere dimension and tangent projectors are used **only for evaluation**.
- These are **not pretrained neural networks** and the transition-mass baseline is **not official LHSD**. Constructed errors do not necessarily satisfy exact cross-noise Gaussian heat-flow consistency.
- Report **selective dimension MAE, accepted exact-dimension accuracy, coverage, paired error differences on commonly accepted anchors, clustered-bootstrap CIs and computational costs**.

The pilot runs on CPU first, then the expanded test. The expanded results use 3 spheres (dimensions 1/2/4), five independent conservative-error realizations, 20 clean anchors per realization, and fixed score-error severities (0, 0.3, 0.8, 1.3). Thus **300 paired test configurations at each severity**.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "numpy", "pandas", "scipy", "matplotlib"])
from pathlib import Path
print('CPU benchmark dependencies installed.')

In [ ]:
%%writefile /content/noise_scale_projector_benchmark.py
#!/usr/bin/env python3
"""Noise-scale selection benchmark: LHSD transition mass vs projector-consistency.

A research prototype, NOT an official LHSD reproduction. A clean on-manifold
anchor is evaluated by the SAME score model at every Gaussian convolution scale.
Oracle geometry and true dimension are used ONLY after both selectors choose.

Default: analytic exact Gaussian-convolved scores + deliberately imperfect,
conservative score potentials. Optional: tiny noise-conditioned denoiser trained
on synthetic continuous spheres; the optional training is not a pretrained model.
"""
import argparse, json, math, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import ive
from scipy.stats import bootstrap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SIGMAS_DEFAULT = [0.012, 0.019, 0.030, 0.047, 0.074, 0.116, 0.181, 0.283, 0.440]
PAPER_C, PAPER_P = 0.1, 4


def make_sphere(d, ambient, seed=0):
    rng=np.random.default_rng(seed)
    U,_=np.linalg.qr(rng.standard_normal((ambient,ambient)))
    return dict(name=f'S^{d} in R^{ambient}', d=d, D=ambient, k=d+1,
                U=U, seed=seed)


def anchors(manifold,n,seed):
    rng=np.random.default_rng(seed)
    us=rng.standard_normal((n,manifold['k']))
    us/=np.linalg.norm(us,axis=1,keepdims=True)
    zs=np.zeros((n,manifold['D']))
    zs[:,:manifold['k']]=us
    xs=zs@manifold['U'].T
    return xs,us


def true_tangent(manifold,u):
    k,D=manifold['k'],manifold['D']
    Q=np.zeros((D,D))
    Q[:k,:k]=np.eye(k)-np.outer(u,u)
    return manifold['U']@Q@manifold['U'].T


def bessel_ratio(nu,t):
    # Stable exp-scaled modified Bessel functions (kappa may be large).
    a=ive(nu+1,t); b=ive(nu,t)
    if b==0 or not np.isfinite(a/b):
        # Stable asymptotic at kappa >> 1, for out-of-support extreme values.
        return 1.-(nu+0.5)/max(t,1e-12)
    return float(a/b)


def oracle_q(manifold,x,sigma):
    """Q = sigma^2 * Hess[-log p_sigma] for uniform sphere + fixed extra coords.

    Uses exact spherical Gaussian convolution, evaluated at ANY x, but the
    benchmark uses original clean on-manifold x only. Analytic normal projector
    in the sigma->0 limit. No training, no estimated network quality.
    """
    U,k,D=manifold['U'],manifold['k'],manifold['D']
    y=U.T@x
    v=y[:k]; r=np.linalg.norm(v)
    if r<1e-12: raise ValueError('radial analytical formula needs nonzero radius')
    n=v/r; kappa=r/sigma**2
    nu=k/2-1
    A=bessel_ratio(nu,kappa)
    # d/dkappa I_{nu+1}(kappa)/I_nu(kappa)
    Aprime=1-A**2-(k-1)*A/kappa
    q_t=1-A/r
    q_r=1-Aprime/(sigma**2)
    Q=np.eye(D)
    Q[:k,:k]=q_t*np.eye(k)+(q_r-q_t)*np.outer(n,n)
    return U@Q@U.T


def conservative_error(manifold,seed=0):
    """Hessian of a quadratic energy perturbation; sym => conservative score."""
    rng=np.random.default_rng(100+seed)
    D=manifold['D']; R=rng.standard_normal((D,D)); R=.5*(R+R.T)
    R/=max(np.linalg.norm(R,ord=2),1e-12)
    return R


def model_q(manifold,x,sigma,kind='analytic',severity=0.0,err=None,
            denoiser=None,device='cpu'):
    if kind=='analytic': return oracle_q(manifold,x,sigma)
    if kind=='controlled':
        # The additive perturbation IS the Hessian of a quadratic energy term.
        # Relative error largest at the lowest sigma; cannot be interpreted as
        # a result about a trained network or used to assert improvement.
        scale=severity*(0.035/sigma)**1.25
        return oracle_q(manifold,x,sigma)+scale*err
    if kind=='learned':
        import torch
        y=torch.tensor(x,dtype=torch.float32,device=device,requires_grad=True)
        ls=torch.tensor([math.log(float(sigma))],dtype=y.dtype,device=device)
        # Differentiate the denoiser D(y,sigma). Q = I-J_D.
        with torch.enable_grad():
            def fun(z): return denoiser(z[None,:],ls[None,:]).reshape(-1)
            J=torch.autograd.functional.jacobian(fun,y,vectorize=True)
        J=J.detach().cpu().numpy().astype(float)
        return np.eye(manifold['D'])-J
    raise ValueError(kind)


def train_denoiser(manifold,sigmas,steps=1400,seed=12,device='cpu'):
    import torch
    from torch import nn
    torch.manual_seed(seed); rng=torch.Generator(device=device).manual_seed(seed)
    D=manifold['D']; k=manifold['k']; U=torch.tensor(manifold['U'],dtype=torch.float32,device=device)
    class Denoiser(nn.Module):
        def __init__(self):
            super().__init__()
            self.net=nn.Sequential(nn.Linear(D+1,128),nn.SiLU(),nn.Linear(128,128),
                                   nn.SiLU(),nn.Linear(128,D))
        def forward(self,y,logsigma):
            return y+torch.exp(logsigma)*self.net(torch.cat([y,logsigma],dim=-1))
    model=Denoiser().to(device); opt=torch.optim.Adam(model.parameters(),lr=1.5e-3)
    sig_lo,sig_hi=float(sigmas[0]),float(sigmas[-1]); hist=[]
    start=time.perf_counter()
    for i in range(steps):
        us=torch.randn(256,k,generator=rng,device=device)
        us=us/us.norm(dim=-1,keepdim=True)
        pad=torch.zeros(256,D,device=device);pad[:,:k]=us
        x=pad@U.T
        logsigma=torch.empty(256,1,device=device).uniform_(math.log(sig_lo),math.log(sig_hi),generator=rng)
        noise=torch.randn(x.shape,generator=rng,device=device)
        y=x+torch.exp(logsigma)*noise
        z=model(y,logsigma)
        # epsilon-prediction objective, normalized uniformly over scales.
        loss=(((z-y)/torch.exp(logsigma)+noise)**2).mean()
        opt.zero_grad(set_to_none=True);loss.backward();opt.step()
        if (i+1)%250==0 or i==0 or i+1==steps:
            print(f'  learned score train {i+1:4d}/{steps}: weighted DSM={loss.item():.4f}')
            hist.append(dict(step=i+1,loss=float(loss.item())))
    model.eval()
    print(f'  training duration {time.perf_counter()-start:.1f}s')
    return model,pd.DataFrame(hist)


def hill(v,c=PAPER_C,p=PAPER_P):
    return 1/(1+(np.abs(v)/c)**p)


def metrics(Q,sigma, c=PAPER_C, p=PAPER_P):
    # Exact spectrum in low D: avoids mixing scale-selection quality with SLQ error.
    sym=.5*(Q+Q.T)
    D=len(Q); eig=np.linalg.eigvalsh(sym)
    norm=np.linalg.norm(sym,'fro')+1e-12
    proj=np.linalg.norm(sym@sym-sym,'fro')/norm
    asym=np.linalg.norm(Q-Q.T,'fro')/(np.linalg.norm(Q,'fro')+1e-12)
    # Same dimension estimator for BOTH selectors: LHSD Hill filter.
    d_eff=float(hill(eig,c,p).sum())
    # LHSD-inspired transition mass, in normalized Q-spectrum; sigma-specific
    # raw H cutoff is c/sigma^2, so the normalized band stays scale-free.
    transition=float(np.mean((np.abs(eig)>=c/2)&(np.abs(eig)<=2*c)))
    # Defect of symmetric spectrum being an orthogonal projector; no true d used.
    rank_est=int(round(d_eff))
    # True projector must have eigenvalues >=0 and <=1; use soft violation metric.
    range_violation=float(np.mean(np.maximum(-eig,0)+np.maximum(eig-1,0)))
    return dict(sigma=float(sigma),transition=transition,projector_defect=float(proj),
                antisymmetry=float(asym),range_violation=range_violation,
                d_eff=d_eff,d_est=rank_est)


def select_interval(rows,selector='baseline'):
    # No ground truth, analytic oracle or true projector accessible here.
    # Fixed criteria across every dataset/seed; no test-label tuning.
    # These diagnostic cutoffs are exploratory: future proposal must calibrate.
    r=sorted(rows,key=lambda z:z['sigma'])
    for i,a in enumerate(r):
        a['cross_scale']=np.nan
        neigh=[]
        if i>0: neigh.append(r[i-1]['Q'])
        if i+1<len(r): neigh.append(r[i+1]['Q'])
        if neigh:
            a['cross_scale']=float(np.mean([
                np.linalg.norm(a['Q']-q,'fro')/(np.linalg.norm(a['Q'],'fro')+1e-12)
                for q in neigh]))
        a['spectral_valid']=bool(a['transition']<=0.12 and 0.5<a['d_eff']<len(a['Q'])-0.5)
        a['joint_valid']=bool(a['spectral_valid'] and a['projector_defect']<=0.24
                              and a['cross_scale']<=0.28 and a['antisymmetry']<=0.06
                              and a['range_violation']<=0.12)
    selected_valid='spectral_valid' if selector=='baseline' else 'joint_valid'
    windows=[]; current=[]
    for i,a in enumerate(r):
        if a[selected_valid]: current.append(i)
        elif current: windows.append(current);current=[]
    if current: windows.append(current)
    # require consecutive 2 scales => no isolated lucky scale. Both methods same rule.
    windows=[w for w in windows if len(w)>=2]
    if not windows:return None
    if selector=='baseline':
        # LHSD-inspired automatic valley: longest low-transition interval;
        # ties break by mean transition mass, then its central scale.
        chosen=min(windows,key=lambda w:(-len(w),np.mean([r[i]['transition'] for i in w])))
        j=chosen[(len(chosen)-1)//2]
    else:
        # Choose safest geometric interval, with no reference to true d.
        def score(i):
            z=r[i]
            return z['projector_defect']+z['cross_scale']+z['transition']
        chosen=min(windows,key=lambda w:(np.mean([score(i) for i in w]),-len(w)))
        j=min(chosen,key=score)
    return r[j]


def project_error(Pest,Ptrue):
    # Normalized Frobenius distance, includes dimension/rank disagreement.
    return float(np.linalg.norm(Pest-Ptrue,'fro')/math.sqrt(max(np.trace(Pest)+np.trace(Ptrue),1e-8)))


def audit_selected(row,Ptrue):
    if row is None:
        return dict(accepted=False,selected_sigma=np.nan,estimated_dim=np.nan,
                    abs_dim_error=np.nan,tangent_projector_error=np.nan,
                    evaluated_sigmas=0, oracle_scale_regret=np.nan)
    Qsym=.5*(row['Q']+row['Q'].T)
    eig,V=np.linalg.eigh(Qsym)
    d=max(0,min(len(eig),int(row['d_est'])))
    idx=np.argsort(np.abs(eig))[:d]
    P=V[:,idx]@V[:,idx].T if d else np.zeros_like(Qsym)
    return dict(accepted=True, selected_sigma=row['sigma'],estimated_dim=d,
                abs_dim_error=float(abs(d-round(np.trace(Ptrue)))),
                tangent_projector_error=project_error(P,Ptrue),
                evaluated_sigmas=0, oracle_scale_regret=np.nan)


def lanczos_slq(Q,probes=8,steps=8,seed=123,c=PAPER_C):
    rng=np.random.default_rng(seed);D=len(Q); vals=[]
    for _ in range(probes):
        v=rng.choice([-1.,1.],size=D); vv=v/np.linalg.norm(v)
        V=[]; alp=[]; bet=[]; old=np.zeros(D); bprev=0.
        for j in range(min(D,steps)):
            V.append(vv.copy()); w=Q@vv
            a=float(vv@w); w-=a*vv+bprev*old
            # full reorthogonalization for stability
            for z in V: w-=float(z@w)*z
            b=float(np.linalg.norm(w));alp.append(a)
            if b<1e-12:break
            if len(alp)<min(D,steps):bet.append(b)
            old=vv;vv=w/b;bprev=b
        T=np.diag(alp)
        for j,b in enumerate(bet): T[j,j+1]=T[j+1,j]=b
        eig,Y=np.linalg.eigh(T)
        vals.append(D*np.sum(Y[0,:]**2*hill(eig,c)))
    return float(np.mean(vals)),float(np.std(vals,ddof=1)/math.sqrt(len(vals)))


def one_config(manifold,model_kind,severity,seed,n,sigmas,denoiser=None,device='cpu'):
    X,us=anchors(manifold,n,seed=30000+seed)
    err=conservative_error(manifold,seed)
    out=[]; all_scales=[]
    for j,(x,u) in enumerate(zip(X,us)):
        Ptrue=true_tangent(manifold,u)
        rows=[]
        for sigma in sigmas:
            Q=model_q(manifold,x,sigma,kind=model_kind,severity=severity,
                      err=err,denoiser=denoiser,device=device)
            rec=metrics(Q,sigma);rec['Q']=Q;rows.append(rec)
        # Ground-truth oracle best scale is computed for EVALUATION ONLY.
        # Selection above and below never sees Ptrue or the correct dimension.
        oracle_best=min(audit_selected(rec,Ptrue)['tangent_projector_error'] for rec in rows)
        base=select_interval(rows,'baseline')
        ours=select_interval(rows,'ours')
        for sel,z in [('LHSD_transition_only',base),('Projector_cross_scale',ours)]:
            audit=audit_selected(z,Ptrue)
            audit['oracle_scale_regret'] = (audit['tangent_projector_error']-oracle_best if audit['accepted'] else np.nan)
            audit['evaluated_sigmas'] = len(sigmas)
            out.append(dict(model=model_kind,severity=severity,shape=manifold['name'],
                            seed=seed,anchor=j,true_dim=manifold['d'],method=sel,
                            **audit))
        for rec in rows:
            all_scales.append(dict(model=model_kind,severity=severity,shape=manifold['name'],
                                   seed=seed,anchor=j,**{k:v for k,v in rec.items() if k!='Q'}))
    return out,all_scales


def summarize(rows):
    df=pd.DataFrame(rows)
    rec=[]
    for (model,sev,method),g in df.groupby(['model','severity','method']):
        a=g[g.accepted]
        # Selective risk = risk conditional on acceptance; never treat missing as zero.
        rec.append(dict(model=model,severity=sev,method=method,n=len(g),
                        coverage=float(g.accepted.mean()),
                        selective_dimension_MAE=float(a.abs_dim_error.mean()) if len(a) else np.nan,
                        exact_dim_rate=float(np.mean(a.abs_dim_error==0)) if len(a) else np.nan,
                        selective_projector_error=float(a.tangent_projector_error.mean()) if len(a) else np.nan,
                        uncond_dimension_MAE=float(g.abs_dim_error.fillna(g.true_dim).mean()),
                        # Penalized dimension MAE is an auxiliary illustrative metric,
                        # not a canonical risk, because abstentions carry arbitrary cost.
                        oracle_scale_regret=float(a.oracle_scale_regret.mean()) if len(a) else np.nan,
                        mean_sigma_evaluations=float(g.evaluated_sigmas.mean()),
                        count_accepted=len(a)))
    return pd.DataFrame(rec)


def paired_comparison(rows):
    df=pd.DataFrame(rows)
    table=df.pivot(index=['model','severity','shape','seed','anchor'],columns='method',
                   values=['accepted','abs_dim_error','tangent_projector_error'])
    table.columns=['_'.join(x) for x in table.columns]
    table=table.reset_index()
    both=table[(table['accepted_LHSD_transition_only']==1)&
               (table['accepted_Projector_cross_scale']==1)].copy()
    if len(both):
        both['paired_dim_gain']=both['abs_dim_error_LHSD_transition_only']-both['abs_dim_error_Projector_cross_scale']
        both['paired_proj_gain']=both['tangent_projector_error_LHSD_transition_only']-both['tangent_projector_error_Projector_cross_scale']
    return table,both


def bootstrap_paired_intervals(rows, n_boot=2500, seed=44):
    # Cluster-resample at the clean image-anchor level, keeping selectors paired.
    # Each configuration is reported separately to avoid mixing severity levels.
    df=pd.DataFrame(rows)
    rng=np.random.default_rng(seed)
    all_reports=[]
    for (model,sev),g in df.groupby(['model','severity']):
        piv=g.pivot(index=['shape','seed','anchor'],columns='method',
                    values=['accepted','abs_dim_error','tangent_projector_error',
                            'oracle_scale_regret'])
        A='LHSD_transition_only'; B='Projector_cross_scale'
        a=piv['accepted'][A].to_numpy().astype(bool)
        b=piv['accepted'][B].to_numpy().astype(bool)
        overlap=a&b
        diffs=(piv['tangent_projector_error'][A].to_numpy(dtype=float)-
               piv['tangent_projector_error'][B].to_numpy(dtype=float))
        gains=diffs[overlap]
        def interval(vals):
            if len(vals)<2:return (np.nan,np.nan)
            ix=rng.integers(0,len(vals),size=(n_boot,len(vals)))
            boot=vals[ix].mean(axis=1)
            return float(np.quantile(boot,.025)),float(np.quantile(boot,.975))
        # Positive delta = lower tangent risk for proposed selector.
        risk_delta=float(gains.mean()) if len(gains) else np.nan
        risk_lo,risk_hi=interval(gains)
        cd=b.astype(float)-a.astype(float)
        c_lo,c_hi=interval(cd)
        all_reports.append(dict(model=model,severity=sev,
            n_pairs=len(piv),both_accepted=int(overlap.sum()),
            paired_projector_risk_gain=risk_delta,
            paired_gain_ci95_low=risk_lo,paired_gain_ci95_high=risk_hi,
            coverage_difference=float(cd.mean()),
            coverage_delta_ci95_low=c_lo,coverage_delta_ci95_high=c_hi,
            limitations='Paired projector gain conditions on both accepting; not a substitute for risk-coverage analysis.'))
    return pd.DataFrame(all_reports)


def visualize(scales,rows,summary,out):
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    dd=pd.DataFrame(rows)
    for method,g in dd.groupby('method'):
        if len(g):
            lab='LHSD transition only' if method.startswith('LHSD') else 'Projector + cross-scale'
            axes[0].bar(lab,g.accepted.mean(),alpha=.8)
    axes[0].set_ylim(0,1);axes[0].set_ylabel('Accepted fraction (coverage)')
    axes[0].set_title('Coverage across all tested anchors')
    for method,g in dd.groupby('method'):
        a=g[g.accepted]
        if len(a):
            lab='LHSD transition only' if method.startswith('LHSD') else 'Projector + cross-scale'
            axes[1].bar(lab,a.tangent_projector_error.mean(),alpha=.8)
    axes[1].set_ylim(bottom=0);axes[1].set_ylabel('Normalized projector error (accepted only)')
    axes[1].set_title('Selective tangent risk (different coverages)')
    fig.tight_layout();fig.savefig(out/'scale_selector_comparison.png',dpi=160);plt.close(fig)
    s=pd.DataFrame(scales)
    fig,axes=plt.subplots(1,3,figsize=(14,4))
    for (shape,model),g in s.groupby(['shape','model']):
        gg=g.groupby('sigma')[['transition','projector_defect','cross_scale']].mean()
        for ax,key in zip(axes,['transition','projector_defect','cross_scale']):
            ax.plot(gg.index,gg[key],marker='o',label=f'{shape}/{model}')
    for ax,title in zip(axes,['LHSD transition mass','Projector idempotence defect','Cross-scale operator drift']):
        ax.set_xscale('log');ax.set_xlabel('Gaussian convolution σ');ax.set_title(title);ax.grid(alpha=.25)
    axes[-1].legend(fontsize=7);fig.tight_layout();fig.savefig(out/'scale_diagnostics.png',dpi=160);plt.close(fig)


def main():
    p=argparse.ArgumentParser()
    p.add_argument('--mode',choices=['pilot','expanded','learned'],default='pilot')
    p.add_argument('--out',default='/content/noise_scale_reliability')
    p.add_argument('--anchors',type=int,default=None)
    p.add_argument('--train-steps',type=int,default=1250)
    p.add_argument('--device',choices=['cpu','cuda'],default='cpu')
    args=p.parse_args()
    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)
    configs=[make_sphere(1,8,1),make_sphere(2,12,2)] if args.mode!='expanded' else [make_sphere(1,8,1),make_sphere(2,12,2),make_sphere(4,16,3)]
    ns=args.anchors or (10 if args.mode=='pilot' else (24 if args.mode=='expanded' else 8))
    all_rows=[];all_scales=[];start=time.perf_counter()
    sigmas=SIGMAS_DEFAULT
    for manifold in configs:
        print(f'\n--- {manifold["name"]} | TRUE d={manifold["d"]} [AUDIT ONLY] ---')
        jobs=[('analytic',0.0,11),('controlled',0.3,11),('controlled',0.8,11)]
        if args.mode=='expanded': jobs += [('controlled',0.3,23),('controlled',0.8,23),('controlled',1.3,11),('controlled',1.3,23)]
        denoiser=None
        if args.mode=='learned':
            denoiser,hist=train_denoiser(manifold,sigmas,args.train_steps,seed=17,device=args.device)
            hist.to_csv(out/f'training_{manifold["d"]}.csv',index=False)
            jobs=[('analytic',0.0,11),('learned',0.0,11)]
        for kind,sev,seed in jobs:
            rows,scales=one_config(manifold,kind,sev,seed,ns,sigmas,
                                    denoiser=denoiser,device=args.device)
            all_rows+=rows;all_scales+=scales
            print(f'  {kind:10s}, severity={sev:.2f}, seed={seed}: completed {ns} anchors')
    dframe=pd.DataFrame(all_rows);sframe=pd.DataFrame(all_scales)
    summ=summarize(all_rows);paired,both=paired_comparison(all_rows)
    dframe.to_csv(out/'per_anchor_selection.csv',index=False)
    sframe.to_csv(out/'all_scale_diagnostics.csv',index=False)
    summ.to_csv(out/'selective_risk_coverage.csv',index=False)
    paired.to_csv(out/'paired_selection.csv',index=False)
    intervals=bootstrap_paired_intervals(all_rows)
    intervals.to_csv(out/'bootstrap_paired_intervals.csv',index=False)
    both.to_csv(out/'both_accepted_paired.csv',index=False)
    visualize(all_scales,all_rows,summ,out)
    # Cost: exact low-D Hessian, sigma sweeps D basis JVPs/scale in learned case.
    runtime=time.perf_counter()-start
    summ.to_string((out/'console_summary.txt').open('w'),index=False)
    with (out/'experiment_metadata.json').open('w') as f:
        json.dump(dict(mode=args.mode,anchors_per_config=ns,noise_scales=sigmas,
                       C=PAPER_C,p=PAPER_P,transition_band='normalized |q| in [c/2, 2c]',
                       runtime_seconds=runtime,selection_uses_truth=False,
                       baseline='LHSD-inspired automated transition-mass-only, NOT official LHSD code',
                       model_notes='analytic exact Gaussian-smoothed score; controlled score is oracle plus Hessian of a quadratic potential; learned is optional model trained from scratch',
                       policy='exploratory constraints not calibrated; no claims of statistically significant improvements',
                       n_test_rows=len(dframe)),f,indent=2)
    print('\nSELECTIVE RISK / COVERAGE (not MNIST results):')
    print(summ.round(4).to_string(index=False))
    print('\nPAIRED BOOTSTRAP (positive risk gain favors proposed on jointly accepted):')
    print(intervals[['model','severity','n_pairs','both_accepted','paired_projector_risk_gain',
                    'paired_gain_ci95_low','paired_gain_ci95_high','coverage_difference']].round(4).to_string(index=False))
    print(f'\nBOTH ACCEPTED: {len(both)} paired anchor configurations; runtime {runtime:.1f} seconds')
    print('Results:',out)

if __name__=='__main__':main()


In [ ]:
%%writefile /content/dimension_accuracy_stress_benchmark.py
#!/usr/bin/env python3
"""Controlled dimension-accuracy stress test for frozen scale selectors.

Uses the *unchanged* LHSD-inspired transition-mass and projector-aware selectors
from noise_scale_projector_benchmark.py, on shared low-D oracle Gaussian-
convolved sphere scores + conservative, sigma-dependent potential perturbations.

These model errors are constructed, NOT a trained score network. The control
baseline is NOT the official LHSD implementation. No ground-truth rank enters
scale selection; it enters metrics only after selection. All output includes
abstentions and paired, common-accepted evaluations.
"""
import argparse, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from noise_scale_projector_benchmark import (
    make_sphere, one_config, SIGMAS_DEFAULT, summarize, paired_comparison,
)

BASE='LHSD_transition_only'
NEW='Projector_cross_scale'
SEVERITIES=[0.0,0.3,0.8,1.3]
MANIFOLD_SPECS=[(1,8,1),(2,12,2),(4,16,3)]
SEEDS=[101,202,303,404,505]


def score_table(df):
    out=[]
    for (sev, method),group in df.groupby(['severity','method']):
        accepted=group[group.accepted]
        out.append(dict(severity=float(sev),method=method,
            n=int(len(group)),accepted_n=int(len(accepted)),
            coverage=float(group.accepted.mean()),
            selective_dimension_MAE=float(accepted.abs_dim_error.mean()) if len(accepted) else float('nan'),
            accepted_exact_dim_accuracy=float(np.mean(accepted.abs_dim_error==0)) if len(accepted) else float('nan'),
            selective_projector_error=float(accepted.tangent_projector_error.mean()) if len(accepted) else float('nan'),
            mean_sigma_evaluations=float(group.evaluated_sigmas.mean())))
    return pd.DataFrame(out)


def paired_by_cluster(df, n_boot=2500, seed=20261101):
    """Cluster-bootstrap at independently perturbed shape-by-seed level.

    A positive paired gain = reduced dimension MAE with the proposed selector,
    conditional on *both methods accepting the same image*. Not a population
    unconditional accuracy claim. Coverage and missingness are also reported.
    """
    piv=df.pivot(index=['severity','shape','seed','anchor'], columns='method',
        values=['accepted','abs_dim_error','tangent_projector_error'])
    piv.columns=['__'.join(item) for item in piv.columns]
    tab=piv.reset_index()
    rng=np.random.default_rng(seed)
    reports=[]; cluster_reports=[]
    for severity,gg in tab.groupby('severity'):
        gg=gg.copy()
        aa=gg['accepted__'+BASE].astype(bool)
        bb=gg['accepted__'+NEW].astype(bool)
        both=aa & bb
        gg['both_accepted']=both
        gg['coverage_change']=bb.astype(float)-aa.astype(float)
        gg['paired_dim_gain']=np.where(both,
            gg['abs_dim_error__'+BASE]-gg['abs_dim_error__'+NEW], np.nan)
        gg['paired_proj_gain']=np.where(both,
            gg['tangent_projector_error__'+BASE]-gg['tangent_projector_error__'+NEW],np.nan)
        cluster=[]
        for (shape,seed_num),g in gg.groupby(['shape','seed']):
            cluster.append(g)
            common=g[g.both_accepted]
            cluster_reports.append(dict(severity=severity,shape=shape,seed=seed_num,
               n=len(g),both_accepted=len(common),
               paired_dim_gain=float(common.paired_dim_gain.mean()) if len(common) else np.nan,
               coverage_change=float(g.coverage_change.mean())))
        # Bootstrap resampling 15 independent shape-by-score-error-realization clusters
        arr=[]
        for _ in range(n_boot):
            ix=rng.integers(len(cluster),size=len(cluster))
            draw=pd.concat([cluster[i] for i in ix],ignore_index=True)
            matched=draw[draw.both_accepted]
            arr.append([
                matched.paired_dim_gain.mean() if len(matched) else np.nan,
                matched.paired_proj_gain.mean() if len(matched) else np.nan,
                draw.coverage_change.mean(),
            ])
        a=np.asarray(arr,dtype=float)
        common=gg[gg.both_accepted]
        def ci(i):
            x=a[:,i];x=x[np.isfinite(x)]
            return (float(np.percentile(x,2.5)),float(np.percentile(x,97.5))) if len(x) else (np.nan,np.nan)
        d_lo,d_hi=ci(0);p_lo,p_hi=ci(1);c_lo,c_hi=ci(2)
        reports.append(dict(severity=float(severity),n_anchors=len(gg),
            independent_shape_error_clusters=len(cluster),n_both_accepted=len(common),
            matched_baseline_dim_MAE=float(common['abs_dim_error__'+BASE].mean()) if len(common) else np.nan,
            matched_ours_dim_MAE=float(common['abs_dim_error__'+NEW].mean()) if len(common) else np.nan,
            matched_paired_dim_MAE_reduction=float(common.paired_dim_gain.mean()) if len(common) else np.nan,
            paired_dim_gain_ci95_low=d_lo,paired_dim_gain_ci95_high=d_hi,
            matched_paired_projector_error_reduction=float(common.paired_proj_gain.mean()) if len(common) else np.nan,
            paired_projector_gain_ci95_low=p_lo,paired_projector_gain_ci95_high=p_hi,
            coverage_difference=float(gg.coverage_change.mean()),
            coverage_delta_ci95_low=c_lo,coverage_delta_ci95_high=c_hi))
    return pd.DataFrame(reports),pd.DataFrame(cluster_reports),tab


def plot_results(summary,paired,out):
    conditions=list(summary.severity.unique())
    fig,axs=plt.subplots(1,3,figsize=(15,4))
    xpos=np.arange(len(conditions));wid=0.35
    for i,(name,method) in enumerate([('Transition-only',BASE),('Projector + cross-scale',NEW)]):
        row=summary[summary.method==method].set_index('severity').loc[conditions]
        axs[0].bar(xpos+(i-.5)*wid,row.selective_dimension_MAE.values,width=wid,label=name)
        axs[1].bar(xpos+(i-.5)*wid,row.coverage.values,width=wid,label=name)
        axs[2].bar(xpos+(i-.5)*wid,row.accepted_exact_dim_accuracy.values,width=wid,label=name)
    for a in axs:
        a.set_xticks(xpos,[f'{x:g}' for x in conditions]);a.set_xlabel('Constructed score-error severity')
        a.grid(axis='y',alpha=.2)
    axs[0].set_ylabel('Dimension MAE (accepted only) ↓')
    axs[1].set_ylabel('Acceptance coverage ↑');axs[1].set_ylim(0,1.05)
    axs[2].set_ylabel('Exact-dimension rate (accepted only) ↑');axs[2].set_ylim(0,1.05)
    axs[0].legend(fontsize=8);fig.suptitle('Controlled score-error stress test — both selectors frozen')
    fig.tight_layout();fig.savefig(out/'dimension_accuracy_and_coverage.png',dpi=160);plt.close(fig)
    fig,ax=plt.subplots(figsize=(8,4.5))
    ax.axhline(0,color='grey',lw=1)
    x=paired.severity.values;y=paired.matched_paired_dim_MAE_reduction.values
    low=y-paired.paired_dim_gain_ci95_low.values;hi=paired.paired_dim_gain_ci95_high.values-y
    ax.errorbar(x,y,yerr=[np.maximum(low,0),np.maximum(hi,0)],fmt='o',capsize=5)
    ax.set_xlabel('Constructed score-error severity');ax.set_ylabel('Matched-pair reduction in dimension MAE ↑')
    ax.set_title('Paired comparison: only same anchors where both accept')
    ax.grid(alpha=.2);fig.tight_layout();fig.savefig(out/'paired_dimension_gain_bootstrap.png',dpi=160);plt.close(fig)


def main():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument('--mode', choices=['pilot','expanded'],default='pilot')
    p.add_argument('--out',default='/content/dimension_accuracy_results')
    p.add_argument('--boot',type=int,default=2500)
    args=p.parse_args()
    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)
    seeds=[101,202] if args.mode=='pilot' else SEEDS
    per_shape=8 if args.mode=='pilot' else 20
    t0=time.perf_counter();rows=[]
    for d,D,ss in MANIFOLD_SPECS:
        shape=make_sphere(d,D,ss)
        for severity in SEVERITIES:
            for seed in seeds:
                records,_=one_config(shape, 'analytic' if severity==0 else 'controlled',
                    severity,seed,per_shape,SIGMAS_DEFAULT)
                rows.extend(records)
        print(f'{shape["name"]}: true dim {d}, finished {len(seeds)*per_shape} anchors per severity',flush=True)
    df=pd.DataFrame(rows)
    # normalize analytic model name for grouping with controlled; keep explicit type
    results=score_table(df)
    paired,clusters,allpair=paired_by_cluster(df,n_boot=args.boot)
    df.to_csv(out/'per_anchor_selection.csv',index=False)
    results.to_csv(out/'dimension_risk_coverage.csv',index=False)
    paired.to_csv(out/'matched_acceptance_paired_bootstrap.csv',index=False)
    clusters.to_csv(out/'per_independent_cluster.csv',index=False)
    allpair.to_csv(out/'all_anchor_paired.csv',index=False)
    plot_results(results,paired,out)
    # Source selection params cannot be trained using held-out true labels.
    runtime=time.perf_counter()-t0
    with (out/'methodology.json').open('w') as f:
        json.dump(dict(protocol='Both methods unchanged from prior noise_scale_projector_benchmark.py',
            mode=args.mode,severities=SEVERITIES,seeds=seeds,anchors_per_shape_seed=per_shape,
            manifolds=MANIFOLD_SPECS,noise_scales=SIGMAS_DEFAULT,
            matrix_computation='exact low-D eigenspectra; no SLQ approximation in scale-selection comparison',
            model_type='analytic Gaussian-smoothed scores or constructed conservative sigma-dependent score error; NOT trained network',
            true_dimensions_used_for_selection=False,baseline='LHSD-inspired simplified transition mass baseline; NOT official LHSD',
            estimator='identical Hill filter and rounding for both selectors, only selected noise scale differs',
            limitations=['constructed scores may not be heat-flow consistent across scales',
                         'selective dimension MAE must be reported with acceptance coverage',
                         'bootstrap independent clusters are shapes x score-error seeds',
                         'no claim of official LHSD superiority or pretrained diffusion model accuracy'],
            runtime_seconds=runtime),f,indent=2)
    print('\nDIMENSION ACCURACY vs COVERAGE — true synthetic dimensions known; no label-based selection:')
    print(results[['severity','method','n','coverage','selective_dimension_MAE','accepted_exact_dim_accuracy','selective_projector_error']].round(4).to_string(index=False))
    print('\nPAIRED ON COMMON ACCEPTED ANCHORS (positive delta favors ours):')
    print(paired[['severity','n_both_accepted','matched_baseline_dim_MAE','matched_ours_dim_MAE','matched_paired_dim_MAE_reduction','paired_dim_gain_ci95_low','paired_dim_gain_ci95_high','coverage_difference']].round(4).to_string(index=False))
    print(f'\nSaved: {out} (runtime {runtime:.1f}s)')

if __name__=='__main__': main()


## A. CPU pilot — small, fast, no neural training

The pilot ensures the full pipeline works; its small cluster count makes confidence intervals unstable. **Use expanded results for your proposal.**

In [ ]:
!python /content/dimension_accuracy_stress_benchmark.py --mode pilot --boot 400 --out /content/dim_accuracy_pilot

## B. Expanded evaluation — proposal-quality controlled benchmark

Run this with the two selectors **unmodified**. Each severity has 300 paired test configurations. The `matched_acceptance_paired_bootstrap.csv` file compares exactly the same accepted anchors and provides intervals by independently resampling the 15 shape × score-error-realization clusters.

**Do not report accuracy without coverage.**

In [ ]:
!python /content/dimension_accuracy_stress_benchmark.py --mode expanded --boot 3000 --out /content/dim_accuracy_expanded

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
folder=Path('/content/dim_accuracy_expanded')
print('Dimension accuracy and coverage:')
display(pd.read_csv(folder/'dimension_risk_coverage.csv'))
print('Paired matched-acceptance analysis with 95% cluster-bootstrap intervals:')
display(pd.read_csv(folder/'matched_acceptance_paired_bootstrap.csv'))
display(Image(filename=str(folder/'dimension_accuracy_and_coverage.png')))
display(Image(filename=str(folder/'paired_dimension_gain_bootstrap.png')))

## C. Interpreting the metrics correctly

- **Dimension MAE (accepted)**: absolute error in the selected estimate against known true sphere dimension, averaging only accepted results.
- **Coverage**: fraction of images that receive a dimension estimate. A method can improve conditional accuracy by rejecting more images; consider both together.
- **Matched-pair dimension improvement**: baseline absolute error minus proposed absolute error **for the same clean image when both accept**. This isolates better scale selection from mere abstention.
- **Cluster-bootstrap confidence interval**: resamples whole shape × score-error-realization clusters, rather than erroneously treating every anchor sharing the same score error as independent.
- **Exact-dimension rate**: fraction of accepted estimates whose rounded dimension is exactly correct.
- **Cost**: both evaluate nine noise scales; the proposed projector and cross-scale diagnostics add arithmetic/HVP cost when scaled to large matrices, **not free**.

**Limits:** Experiments involve exact low-D matrices and deliberately constructed (conservative) score errors. Our prior test of a small trained score network did **not** demonstrate the same advantage; official LHSD + pretrained diffusion validation remains future research. The tested effect may not transfer to higher dimensional manifolds; do not claim SOTA.

## D. Download the result package

Use the ZIP for the proposal table and figures. To repeat, rerun the exact notebook; results are seed-controlled.

In [ ]:
import shutil
archive=shutil.make_archive('/content/dimension_accuracy_expanded_results','zip','/content/dim_accuracy_expanded')
print('Saved',archive)
# Optional download on desktop:
# from google.colab import files; files.download(archive)